### 链的高级使用
---
* 使用@chain修饰符快速将函数变为链
* 在链中使用lambda函数
* 在链中自定义支持流输出的函数
* 使用RunnablePassthrough在链中传值
* 如何在运行时动态添加链的配置
* 为链增加记忆(v0.2)
* 创建路由链
* 为链添加回退机制

In [1]:
# 导入大模型
from langchain_deepseek import ChatDeepSeek
# 导入字符串输出解析器
from langchain_core.output_parsers import StrOutputParser
# 导入chain装饰器，用于创建自定义链
from langchain_core.runnables import chain
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv
load_dotenv()

# 初始化DeepSeek大语言模型
model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

# 创建第一个提示模板：请求关于特定主题的笑话
prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
# 创建第二个提示模板：询问笑话的主题是什么
prompt2 = ChatPromptTemplate.from_template("What is the subject of this joke: {joke}")

# 使用@chain装饰器定义一个自定义链
@chain
def custom_chain(text):
    # 步骤1：将输入文本填充到第一个提示模板中
    prompt_val1 = prompt1.invoke({"topic": text})
    # 步骤2：使用DS模型生成关于指定主题的笑话
    output1 = model.invoke(prompt_val1)
    # 步骤3：将模型输出解析为字符串
    parsed_output1 = StrOutputParser().invoke(output1)

    # 步骤4：创建第二个处理链，用于分析笑话主题
    # 这个链将提示模板、DS模型和字符串解析器串联起来
    chain2 = prompt2 | model | StrOutputParser()

    # 步骤5：将第一步生成的笑话作为输入，让第二个链分析其主题
    return chain2.invoke({"joke": parsed_output1})

# 调用自定义链，输入主题“bears”（熊）
# 整个过程：
# 1. 先生成一个关于熊的笑话
# 2. 然后分析这个笑话的主题是什么
# 3. 返回分析结果
custom_chain.invoke("bears")


'The subject is **bears—specifically why bears have hairy coats**. The joke is a pun: **“fur protection”** sounds like **“for protection.”**'

#### 使用函数

In [4]:
from operator import itemgetter # 导入itemgetter函数，用于从字典中提取值

from langchain_core.prompts import ChatPromptTemplate # 导入聊天提示模板
from langchain_core.runnables import RunnableLambda # 导入可运行的Lambda函数包装器
from langchain_deepseek import ChatDeepSeek # 导入DeepSeek聊天模型
import os
from dotenv import load_dotenv

load_dotenv()

# 初始化DeepSeek大语言模型
model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

def length_fuction(text):
    return len(text)

def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text1"], _dict["text2"])

# 创建一个简单的聊天提示模板，询问a和b的和
prompt = ChatPromptTemplate.from_template("What is {a} + {b}")

# 构建一个复杂的处理链
chain = (
    {
        # 处理"a"参数：
        # 1. 从输入字典中提取"foo"键的值
        # 2. 将提取的值传递给length_function函数（假设这个函数计算字符串长度）
        "a": itemgetter("foo") | RunnableLambda(length_fuction),

        # 处理"b"参数：
        # 1. 创建一个包含两个键值对的字典：
        #    - "text1": 从输入字典中提取“foo”键的值
        #    - "text2": 从输入字典中提取“bar”键的值
        # 2. 将这个字典传递给multiple_length_function函数
        "b": {"text1": itemgetter("foo"), "text2": itemgetter("bar")} 
        | RunnableLambda(multiple_length_function),
    } 
    | prompt # 将处理后的"a"和"b"值填入提示模板
    | model  # 将填充后的提示发送给DeepSeek模型生成回答
)

# 调用链处理流程，输入一个包含“foo”和“bar”键的字典
# 整个过程：
# 1. 计算“bar”字符串的长度作为a的值
# 2. 计算“bar”和“gah”字符串的总长度作为b的值
# 3. 将这些值填入提示"What is {a} + {b}"
# 4. 让DeepSeek模型回答这个问题
chain.invoke({"foo":"bar", "bar": "gah"})

AIMessage(content='12', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple. Must not mention? "12". Ensure concise.'}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 37, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 16, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 37}, 'model_provider': 'deepseek', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a1a5b3f2972e5b5497f1d936336840', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a5b3-e998-7ec0-8a3a-77f8f4555a7e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 17, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 16}